[← Back to 07 - Window Functions, Part 2](<../07-window-functions-part-2.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 07 - Window Functions, Part 2: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/07-window-functions-part-2-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 1. Four customers, three buckets

In [3]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
)
SELECT
    customer_name,
    total_spent,
    NTILE(3) OVER (ORDER BY total_spent DESC) AS spend_tier
FROM customer_totals
ORDER BY
    total_spent DESC;

customer_name,total_spent,spend_tier
Bola,170,1
Chidi,140,1
Ada,100,2
Dami,0,3


Bucket 1 gets the extra customer, and Dami sits alone in bucket 3. SQLite's documentation for `ntile()` puts the rule in four words: "larger groups occur first." With 4 rows and 3 buckets, the sizes come out 2, 1, 1, so the top tier here is easier to get into than either tier below it. On a few thousand rows the difference is one row and nobody notices; on a small group like this, it's worth saying out loud in whatever report uses it.

## 2. A centered moving average

In [4]:
%%sql
SELECT
    order_id,
    amount,
    ROUND(AVG(amount) OVER (
        ORDER BY order_id
        ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
    ), 2) AS centered_avg,
    COUNT(*) OVER (
        ORDER BY order_id
        ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
    ) AS rows_in_frame
FROM orders
ORDER BY
    order_id;

order_id,amount,centered_avg,rows_in_frame
101,40,50.0,2
102,60,60.0,3
103,80,76.67,3
104,90,103.33,3
105,140,115.0,2


The first and last orders average only two rows each, because there's nothing before order 101 and nothing after order 105. A centered window has a short frame at *both* edges, where a trailing one (section 4) only has it at the start. Centered averages are common for smoothing a chart after the fact; trailing ones are what you'd use when each row may only look at what had already happened by then.

## 3. Distance from the last order

In [5]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    amount - LAST_VALUE(amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS vs_last_order
FROM orders
ORDER BY
    customer_id,
    order_id;

customer_id,order_id,amount,vs_last_order
1,101,40,-20
1,102,60,0
2,103,80,-10
2,104,90,0
3,105,140,0


Ada's first order is 20 smaller than her last, Bola's is 10 smaller, and every most-recent order shows 0. Without the explicit frame, every row would show 0, since `LAST_VALUE()` would hand back the current row's own amount: section 7's trap, which in a subtraction looks like "no change" rather than like a bug.

## 4. Stretch: a three-month moving average for 2012 (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [6]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [7]:
%%tsql
WITH monthly_sales AS (
    SELECT
        DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1) AS month_start,
        SUM(TotalDue) AS sales
    FROM Sales.SalesOrderHeader
    GROUP BY
        DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1)
),
moving AS (
    SELECT
        month_start,
        sales,
        AVG(sales) OVER (
            ORDER BY month_start
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS three_month_avg,
        COUNT(*) OVER (
            ORDER BY month_start
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rows_in_frame
    FROM monthly_sales
)
SELECT
    month_start,
    sales,
    three_month_avg,
    rows_in_frame
FROM moving
WHERE month_start >= '2012-01-01'
  AND month_start < '2013-01-01'
ORDER BY
    month_start;

month_start,sales,three_month_avg,rows_in_frame
2012-01-01,4458337.4444,2245366.4527,3
2012-02-01,1649051.9001,2523279.4143,3
2012-03-01,3336347.4716,3147912.2720,3
2012-04-01,1871923.5039,2285774.2918,3
2012-05-01,3452924.4537,2887065.1430,3
2012-06-01,4610647.2153,3311831.7243,3
2012-07-01,3840231.4590,3967934.3760,3
2012-08-01,2442451.1831,3631109.9524,3
2012-09-01,3881724.1860,3388135.6093,3
2012-10-01,2858060.1970,3060745.1887,3


Every 2012 month averages three full months, January included, because the moving average is computed over *all* months in the `moving` CTE before `WHERE` keeps only 2012. Filter first and January would average only itself, February two months, and the year would start with two misleading numbers, the same lesson as the January `NULL` in `06`'s month-over-month query: a window can only see the rows that survive `WHERE`.

---

[← Back to 07 - Window Functions, Part 2](<../07-window-functions-part-2.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).